# Robust Court Citation Concept Enrichment Smoke Test

This notebook enriches Swiss court citation rows from `court_considerations.csv` using a compact, query-neutral legal descriptor schema.

Design goals:

- Works with **any citation + text** pair, including short, long, procedural, constitutional, civil, criminal, tax, planning, and administrative-law citations.
- Does **not** generate synthetic user questions.
- Does **not** generate generic summaries.
- Produces compact concepts/topics/keywords that make each citation specific.
- Uses vLLM structured/guided JSON decoding when available.
- Retries failed generations one-by-one.
- Repairs truncated/messy JSON where possible.
- Always writes an output row; if LLM parsing fails after retries, it emits a clearly marked deterministic fallback row instead of crashing.


## Notes for 2×T4

Your current run used only GPU 0 because `tensor_parallel_size=1`. That is expected.

For this 10-row smoke test, use `gpu_mode = "single"`. For using both T4s inside one notebook, set `gpu_mode = "tp2"`, but that shards one model across both GPUs and may not be fastest on Kaggle.

For production throughput, the better approach is two independent Kaggle sessions/notebooks:
- Notebook A: `gpu_mode="single"`, `CUDA_VISIBLE_DEVICES=0`, `start=0`, `limit=...`
- Notebook B: `gpu_mode="single"`, `CUDA_VISIBLE_DEVICES=0` inside the second session, with a different `start`

If staying in one notebook, TP=2 is acceptable for testing but not necessarily the highest cards/second.


In [ ]:
# Cell 1 - Install/import dependencies

import os
import re
import gc
import ast
import json
import time
import traceback
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from collections import Counter

import pandas as pd
from tqdm.auto import tqdm

try:
    import torch
except Exception:
    torch = None

print("Python imports OK")
if torch is not None:
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(i)
            free, total = torch.cuda.mem_get_info(i)
            print(f"GPU {i}: {props.name} total={total/1024**3:.2f} GiB free={free/1024**3:.2f} GiB")


In [ ]:
# Cell 2 - Configuration

@dataclass
class Config:
    # Kaggle competition CSV
    input_csv: str = "/kaggle/input/competitions/llm-agentic-legal-information-retrieval/court_considerations.csv"
    fallback_input_csv: str = "court_considerations.csv"

    # Local Kaggle model path supplied by user
    model_name: str = "/kaggle/input/models/qwen-lm/qwen-3/transformers/8b-awq/1"

    output_dir: str = "/kaggle/working"
    output_jsonl: str = "enriched_court_citations_10.jsonl"
    output_preview_csv: str = "enriched_court_citations_10_preview.csv"
    output_failures_jsonl: str = "enriched_court_citations_10_failures.jsonl"

    # Input selection
    n_rows: int = 10
    sample_random: bool = False
    random_seed: Optional[int] = 42
    min_text_chars: int = 250
    max_text_chars: int = 4500

    # Manual single/multiple citation mode.
    # If manual_records is non-empty, the notebook ignores the CSV.
    manual_records: List[Dict[str, str]] = None

    # Engine
    engine: str = "vllm"  # "vllm" or "transformers"

    # GPU mode:
    # - "single": one model on GPU 0. Best for smoke test.
    # - "tp2": one tensor-parallel model across both T4s. Uses both GPUs but may not be fastest.
    # - For true production speed, run two notebook copies with mode="single" and different start/limit shards.
    gpu_mode: str = "single"

    # Sharding for production runs
    start: int = 0
    limit: int = 10

    # vLLM settings
    tensor_parallel_size: int = 1
    gpu_memory_utilization: float = 0.78
    max_model_len: int = 4096
    enforce_eager: bool = True
    max_num_seqs: int = 8
    quantization: str = "awq_marlin"  # faster than explicit awq on this model/vLLM
    disable_custom_all_reduce: bool = True
    force_triton_attention: bool = True

    # IMPORTANT:
    # Your Kaggle vLLM v0.20 run failed with:
    # AttributeError("'dict' object has no attribute '_backend'")
    # when using structured_outputs. Therefore default to prompt-constrained JSON
    # plus robust parser/repair. Keep False unless you know your vLLM build accepts it.
    use_structured_outputs: bool = False

    # Generation
    batch_size: int = 4
    max_new_tokens: int = 768
    retry_max_new_tokens: int = 1024
    temperature: float = 0.0
    top_p: float = 1.0
    repetition_penalty: float = 1.02
    enable_thinking: bool = False
    max_retries: int = 2

cfg = Config(manual_records=[])

# Apply GPU mode
if cfg.gpu_mode == "single":
    # Must be set before vLLM initializes. If you already loaded vLLM, restart the kernel.
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
    cfg.tensor_parallel_size = 1
elif cfg.gpu_mode == "tp2":
    # Uses both T4s for one sharded model.
    os.environ.pop("CUDA_VISIBLE_DEVICES", None)
    cfg.tensor_parallel_size = 2
    cfg.disable_custom_all_reduce = True
    # TP=2 has more communication overhead on Kaggle T4s, so keep batch conservative.
    cfg.max_num_seqs = min(cfg.max_num_seqs, 8)
else:
    raise ValueError("cfg.gpu_mode must be 'single' or 'tp2'")

# Keep cfg.n_rows aligned with cfg.limit for smoke tests.
cfg.n_rows = cfg.limit if cfg.limit else cfg.n_rows

print(asdict(cfg))

model_path = Path(cfg.model_name)
if not model_path.exists():
    raise FileNotFoundError(
        f"Model path does not exist: {model_path}\n"
        "Attach the Kaggle model dataset or update cfg.model_name."
    )
print("Model path OK:", model_path)


In [ ]:
# Cell 3 - Resolve input and select rows

def resolve_input_path(cfg: Config) -> Path:
    candidates = [
        Path(cfg.input_csv),
        Path(cfg.fallback_input_csv),
        Path("/kaggle/working") / cfg.fallback_input_csv,
        Path("/mnt/data") / cfg.fallback_input_csv,
        Path("/mnt/data") / "court_consideration.csv",
        Path("/mnt/data") / "court_considerations.csv",
    ]
    for p in candidates:
        if p.exists():
            return p

    if Path("/kaggle/input").exists():
        patterns = ["**/court_considerations.csv", "**/court_consideration.csv"]
        for pat in patterns:
            found = sorted(Path("/kaggle/input").glob(pat))
            if found:
                return found[0]

    raise FileNotFoundError("Could not find court_considerations.csv. Update cfg.input_csv.")

def pick_column(columns: List[str], preferred: List[str], contains_any: List[str]) -> Optional[str]:
    lower_map = {c.lower(): c for c in columns}
    for name in preferred:
        if name.lower() in lower_map:
            return lower_map[name.lower()]
    for c in columns:
        lc = c.lower()
        if any(token in lc for token in contains_any):
            return c
    return None

def build_work_df_from_csv(cfg: Config) -> Tuple[pd.DataFrame, str, str]:
    input_path = resolve_input_path(cfg)
    print("Using input:", input_path)

    df = pd.read_csv(input_path)
    print("Shape:", df.shape)
    print("Columns:", list(df.columns))

    columns = list(df.columns)
    citation_col = pick_column(
        columns,
        preferred=["citation", "cite", "court_citation", "authority_citation", "consideration_citation"],
        contains_any=["citation", "cite", "bge"],
    )
    text_col = pick_column(
        columns,
        preferred=["text", "consideration_text", "paragraph_text", "content", "raw_text", "body"],
        contains_any=["text", "content", "paragraph", "consideration", "body"],
    )

    if citation_col is None:
        raise ValueError("Could not infer citation column. Set citation_col manually.")
    if text_col is None:
        raise ValueError("Could not infer text column. Set text_col manually.")

    print("citation_col:", citation_col)
    print("text_col:", text_col)

    valid = df[df[citation_col].notna() & df[text_col].notna()].copy()
    valid[text_col] = valid[text_col].astype(str)
    valid["_text_len"] = valid[text_col].str.strip().str.len()

    # Prefer substantive rows for smoke testing, but still allow shorter rows if the dataset has them.
    substantive = valid[valid["_text_len"] >= cfg.min_text_chars].copy()
    if len(substantive) < cfg.n_rows:
        print(f"Only {len(substantive)} rows above min_text_chars={cfg.min_text_chars}; falling back to all non-empty rows.")
        substantive = valid[valid["_text_len"] > 30].copy()

    if cfg.sample_random:
        pool = substantive
        if cfg.start:
            pool = pool.iloc[cfg.start:]
        work_df = pool.sample(n=min(cfg.n_rows, len(pool)), random_state=cfg.random_seed)
    else:
        # Avoid tiny fragments through min_text_chars, then allow deterministic sharding.
        end = None if not cfg.limit else cfg.start + cfg.limit
        work_df = substantive.iloc[cfg.start:end]

    work_df = work_df.reset_index(drop=False).rename(columns={"index": "_source_row"})
    print("Selected rows:", len(work_df))
    display(work_df[[citation_col, text_col, "_text_len"]].head(cfg.n_rows))
    return work_df, citation_col, text_col

def build_work_df_from_manual(records: List[Dict[str, str]]) -> Tuple[pd.DataFrame, str, str]:
    rows = []
    for i, rec in enumerate(records):
        rows.append({
            "_source_row": i,
            "citation": str(rec.get("citation", "")),
            "text": str(rec.get("text", "")),
            "_text_len": len(str(rec.get("text", ""))),
        })
    work_df = pd.DataFrame(rows)
    print("Using manual_records:", len(work_df))
    display(work_df[["citation", "text", "_text_len"]].head(20))
    return work_df, "citation", "text"

if cfg.manual_records:
    work_df, citation_col, text_col = build_work_df_from_manual(cfg.manual_records)
else:
    work_df, citation_col, text_col = build_work_df_from_csv(cfg)


In [ ]:
# Cell 4 - Compact schema, enums, prompt rules

ROLE_VALUES = {
    "holding", "reasoning", "facts", "procedural_history", "citation", "dissent", "neutral"
}

OUTCOME_VALUES = {
    "granted", "dismissed", "inadmissible", "remitted", "partial", "neutral", "none"
}

AUTHORITY_VALUES = {
    "leading_decision",
    "settled_rule",
    "legal_test",
    "standard_of_review",
    "constitutional_standard",
    "statutory_interpretation",
    "application_of_rule",
    "distinguishing_case",
    "procedural_background",
    "factual_background",
    "background",
    "none",
}

GENERIC_CONCEPTS = {
    "law", "legal", "court", "decision", "case", "appeal", "judgment", "procedure",
    "rights", "claim", "application", "review", "authority", "public law", "private law"
}

ENRICHMENT_JSON_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "properties": {
        "legal_area": {"type": "string", "maxLength": 60},
        "legal_domain_path": {
            "type": "array",
            "items": {"type": "string", "maxLength": 45},
            "minItems": 2,
            "maxItems": 6,
        },
        "topic": {"type": "string", "maxLength": 80},
        "subtopic": {"type": "string", "maxLength": 90},
        "micro_topic": {"type": "string", "maxLength": 130},
        "concepts_en": {
            "type": "array",
            "items": {"type": "string", "maxLength": 60},
            "minItems": 3,
            "maxItems": 8,
        },
        "terms_original": {
            "type": "array",
            "items": {"type": "string", "maxLength": 80},
            "maxItems": 10,
        },
        "statute_anchors": {
            "type": "array",
            "items": {"type": "string", "maxLength": 80},
            "maxItems": 8,
        },
        "case_anchors": {
            "type": "array",
            "items": {"type": "string", "maxLength": 90},
            "maxItems": 8,
        },
        "doctrinal_rule": {"type": "string", "maxLength": 240},
        "legal_test": {"type": "string", "maxLength": 200},
        "fact_pattern_tags": {
            "type": "array",
            "items": {"type": "string", "maxLength": 60},
            "maxItems": 6,
        },
        "procedural_context": {"type": "string", "maxLength": 90},
        "paragraph_role": {"type": "string", "enum": sorted(ROLE_VALUES)},
        "authority_role": {
            "type": "array",
            "items": {"type": "string", "enum": sorted(AUTHORITY_VALUES)},
            "minItems": 1,
            "maxItems": 4,
        },
        "outcome_signal": {"type": "string", "enum": sorted(OUTCOME_VALUES)},
        "specificity_score": {"type": "number", "minimum": 0, "maximum": 1},
    },
    "required": [
        "legal_area",
        "legal_domain_path",
        "topic",
        "subtopic",
        "micro_topic",
        "concepts_en",
        "terms_original",
        "statute_anchors",
        "case_anchors",
        "doctrinal_rule",
        "legal_test",
        "fact_pattern_tags",
        "procedural_context",
        "paragraph_role",
        "authority_role",
        "outcome_signal",
        "specificity_score",
    ],
}

JSON_SCHEMA_TEXT = json.dumps(ENRICHMENT_JSON_SCHEMA, ensure_ascii=False, indent=2)

SYSTEM_PROMPT = """You are a deterministic Swiss legal citation enrichment engine.

Return exactly one compact valid JSON object matching the schema.
No markdown. No explanations. No generated user questions. No summary field.

Your job is not to predict user questions.
Your job is to create a query-neutral legal fingerprint:
legal area, topic, subtopic, micro-topic, concepts, exact original-language terms, anchors, rule/test, fact pattern, procedural context, role, authority role, outcome signal.

Rules:
- Use English for classification fields.
- terms_original must use exact important terms from the source text language.
- statute_anchors and case_anchors must include only citations/articles explicitly present in the text.
- Do not include the input citation itself in case_anchors unless the text itself cites it separately.
- Prefer specific concepts over generic words.
- Keep arrays short and high-signal.
- Keep doctrinal_rule and legal_test concise.
- If the text is fragmentary, produce the best grounded descriptors and set paragraph_role to citation/procedural_history/neutral when appropriate.
- Never invent article numbers, case citations, procedural outcomes, or facts.
""".strip()

USER_TEMPLATE = """Citation:
{citation}

Text:
{text}

JSON schema:
{schema}

Return JSON only.
""".strip()

def trim_text(text: str, max_chars: int) -> str:
    text = re.sub(r"\s+", " ", str(text)).strip()
    if len(text) <= max_chars:
        return text
    # Preserve beginning and end because citations/statutes can appear near either side.
    head = max_chars // 2
    tail = max_chars - head
    return text[:head].rstrip() + " ... [TRUNCATED] ... " + text[-tail:].lstrip()

def build_prompt(row: Dict[str, Any], cfg: Config, repair: bool = False, bad_output: str = "", error: str = "") -> str:
    citation = str(row["citation"]).strip()
    text = trim_text(row["text"], cfg.max_text_chars)

    if not repair:
        return USER_TEMPLATE.format(citation=citation, text=text, schema=JSON_SCHEMA_TEXT)

    return f"""The previous output was invalid JSON or violated the schema.

Citation:
{citation}

Text:
{text}

Previous invalid output:
{bad_output[:2500]}

Parser/validation error:
{error}

Repair instructions:
- Return exactly one complete valid compact JSON object.
- Match the schema exactly.
- Do not include markdown.
- Do not include generated user questions.
- Do not include summary_en or query_phrases_en.
- Keep arrays short.

JSON schema:
{JSON_SCHEMA_TEXT}

Return repaired JSON only.
""".strip()


In [ ]:
# Cell 5 - Parsing, normalization, validation, fallback, retrieval views

def find_balanced_json_object(s: str) -> str:
    """Extract the first balanced JSON object from model output."""
    if s is None:
        raise ValueError("Empty model output")

    s = str(s).strip()
    # Remove code fences if the model ignored instructions.
    s = re.sub(r"^\s*```(?:json)?\s*", "", s, flags=re.I)
    s = re.sub(r"\s*```\s*$", "", s)

    start = s.find("{")
    if start < 0:
        raise ValueError(f"No JSON object start found: {s[:300]}")

    depth = 0
    in_str = False
    escape = False
    for i in range(start, len(s)):
        ch = s[i]
        if in_str:
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == '"':
                in_str = False
        else:
            if ch == '"':
                in_str = True
            elif ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    return s[start:i+1]
    raise ValueError(f"No balanced JSON object found: {s[:700]}")

def parse_json_lenient(raw: str) -> Dict[str, Any]:
    js = find_balanced_json_object(raw)

    # First strict JSON.
    try:
        return json.loads(js)
    except Exception:
        pass

    # Common cleanup.
    cleaned = js
    cleaned = re.sub(r",\s*([}\]])", r"\1", cleaned)
    cleaned = cleaned.replace("\u201c", '"').replace("\u201d", '"').replace("\u2018", "'").replace("\u2019", "'")

    try:
        return json.loads(cleaned)
    except Exception:
        pass

    # Last resort: Python literal if valid-ish.
    try:
        return ast.literal_eval(cleaned)
    except Exception as exc:
        raise ValueError(f"Could not parse JSON object: {repr(exc)}; raw={raw[:800]}")

def dedupe_keep_order(items: List[Any]) -> List[str]:
    out, seen = [], set()
    for x in items or []:
        if x is None:
            continue
        s = re.sub(r"\s+", " ", str(x)).strip()
        if not s:
            continue
        key = s.casefold()
        if key not in seen:
            seen.add(key)
            out.append(s)
    return out

def clamp_string(s: Any, max_len: int = 240) -> str:
    s = re.sub(r"\s+", " ", str(s or "")).strip()
    return s[:max_len].rstrip()

def normalize_enum(value: Any, allowed: set, default: str) -> str:
    v = str(value or "").strip().lower().replace(" ", "_").replace("-", "_")
    aliases = {
        "remanded": "remitted",
        "partially_granted": "partial",
        "none_or_neutral": "neutral",
        "not_applicable": "none",
        "procedural": "procedural_history",
        "analysis": "reasoning",
        "rule_application": "application_of_rule",
        "application": "application_of_rule",
        "legal_standard": "legal_test",
        "standard": "legal_test",
        "constitutional": "constitutional_standard",
        "statutory": "statutory_interpretation",
        "factual": "factual_background",
        "procedural": "procedural_background",
    }
    v = aliases.get(v, v)
    return v if v in allowed else default

def normalize_authority_roles(values: Any) -> List[str]:
    if isinstance(values, str):
        values = [values]
    vals = []
    for v in values or []:
        vals.append(normalize_enum(v, AUTHORITY_VALUES, "none"))
    vals = dedupe_keep_order(vals)
    if not vals:
        vals = ["none"]
    if "none" in vals and len(vals) > 1:
        vals = [v for v in vals if v != "none"]
    return vals[:4]

def citation_base(citation: str) -> str:
    # "BGE 145 I 1 E. 6.5.1" -> "BGE 145 I 1"
    m = re.match(r"^\s*(BGE\s+\d+\s+[IVXLC]+\s+\d+)", citation or "", flags=re.I)
    return m.group(1).strip() if m else str(citation or "").strip()

def remove_self_case_anchor(case_anchors: List[str], citation: str) -> List[str]:
    cit = str(citation or "").strip().casefold()
    base = citation_base(citation).casefold()
    out = []
    for a in case_anchors:
        aa = a.casefold().strip()
        # Remove exact current citation and exact court_base self-reference.
        if aa == cit or aa == base:
            continue
        out.append(a)
    return out

def extract_statutes_from_text(text: str) -> List[str]:
    text = str(text or "")
    # Swiss-style articles: Art. 34 Abs. 2 BV, Art. 221 StPO, etc.
    pat = r"\bArt\.\s*\d+[a-zA-Z]*\s*(?:Abs\.\s*\d+[a-zA-Z]*)?\s*(?:lit\.\s*[a-z])?\s*(?:[A-ZÄÖÜ][A-Za-zÄÖÜäöü]{1,10})?"
    return dedupe_keep_order(re.findall(pat, text))[:8]

def extract_case_citations_from_text(text: str, current_citation: str) -> List[str]:
    text = str(text or "")
    patterns = [
        r"\bBGE\s+\d+\s+[IVXLC]+\s+\d+(?:\s+E\.\s*[\d.]+)?",
        r"\bUrteil\s+\d+[A-Z_]*\s+\d+/\d{4}",
    ]
    found = []
    for pat in patterns:
        found.extend(re.findall(pat, text))
    return remove_self_case_anchor(dedupe_keep_order(found)[:12], current_citation)[:8]

def normalize_enrichment(obj: Dict[str, Any], citation: str, text: str) -> Dict[str, Any]:
    # Fill missing with safe defaults.
    out = {}
    out["legal_area"] = clamp_string(obj.get("legal_area", "unspecified legal area"), 60)

    path = dedupe_keep_order(obj.get("legal_domain_path", []))[:6]
    if len(path) < 2:
        path = dedupe_keep_order([out["legal_area"], obj.get("topic", "unspecified topic")])[:6]
    out["legal_domain_path"] = [clamp_string(x, 45) for x in path]

    out["topic"] = clamp_string(obj.get("topic", ""), 80) or "unspecified topic"
    out["subtopic"] = clamp_string(obj.get("subtopic", ""), 90) or out["topic"]
    out["micro_topic"] = clamp_string(obj.get("micro_topic", ""), 130) or out["subtopic"]

    concepts = dedupe_keep_order(obj.get("concepts_en", []))
    concepts = [c for c in concepts if c.casefold() not in GENERIC_CONCEPTS]
    out["concepts_en"] = [clamp_string(x, 60) for x in concepts[:8]]
    if len(out["concepts_en"]) < 3:
        # Use topic fields as a grounded fallback if the model was too sparse.
        out["concepts_en"] = dedupe_keep_order([out["topic"], out["subtopic"], out["micro_topic"]])[:8]

    out["terms_original"] = [clamp_string(x, 80) for x in dedupe_keep_order(obj.get("terms_original", []))[:10]]

    statutes = dedupe_keep_order(obj.get("statute_anchors", []))
    # Add deterministic extractions from text, then dedupe.
    statutes = dedupe_keep_order(statutes + extract_statutes_from_text(text))[:8]
    out["statute_anchors"] = [clamp_string(x, 80) for x in statutes]

    cases = dedupe_keep_order(obj.get("case_anchors", []))
    cases = dedupe_keep_order(cases + extract_case_citations_from_text(text, citation))
    cases = remove_self_case_anchor(cases, citation)[:8]
    out["case_anchors"] = [clamp_string(x, 90) for x in cases]

    out["doctrinal_rule"] = clamp_string(obj.get("doctrinal_rule", ""), 240)
    out["legal_test"] = clamp_string(obj.get("legal_test", ""), 200)
    out["fact_pattern_tags"] = [clamp_string(x, 60) for x in dedupe_keep_order(obj.get("fact_pattern_tags", []))[:6]]
    out["procedural_context"] = clamp_string(obj.get("procedural_context", ""), 90)

    out["paragraph_role"] = normalize_enum(obj.get("paragraph_role"), ROLE_VALUES, "neutral")
    out["authority_role"] = normalize_authority_roles(obj.get("authority_role", ["none"]))
    out["outcome_signal"] = normalize_enum(obj.get("outcome_signal"), OUTCOME_VALUES, "none")

    try:
        score = float(obj.get("specificity_score", 0.5))
    except Exception:
        score = 0.5
    out["specificity_score"] = max(0.0, min(1.0, score))

    # No forbidden fields.
    for forbidden in ["query_phrases_en", "summary_en", "natural_language_queries", "english_summary"]:
        out.pop(forbidden, None)

    return out

def validate_enrichment(enr: Dict[str, Any]) -> List[str]:
    errors = []
    required = list(ENRICHMENT_JSON_SCHEMA["required"])
    for k in required:
        if k not in enr:
            errors.append(f"missing:{k}")

    if enr.get("paragraph_role") not in ROLE_VALUES:
        errors.append("bad_paragraph_role")
    if enr.get("outcome_signal") not in OUTCOME_VALUES:
        errors.append("bad_outcome_signal")
    if not enr.get("concepts_en"):
        errors.append("empty_concepts_en")
    if not enr.get("topic") or not enr.get("subtopic"):
        errors.append("empty_topic")
    forbidden = set(enr.keys()) & {"query_phrases_en", "summary_en", "natural_language_queries", "english_summary"}
    if forbidden:
        errors.append(f"forbidden_fields:{sorted(forbidden)}")
    return errors

def deterministic_fallback(citation: str, text: str, error: str) -> Dict[str, Any]:
    statutes = extract_statutes_from_text(text)
    cases = extract_case_citations_from_text(text, citation)

    # Small phrase heuristic for original terms: capitalized German/French/Italian legal-ish nouns.
    words = re.findall(r"\b[A-ZÄÖÜ][A-Za-zÄÖÜäöüéèàùçîïô]{5,}\b", str(text or ""))
    terms = dedupe_keep_order(words)[:10]

    fallback = {
        "legal_area": "unspecified legal area",
        "legal_domain_path": ["unspecified legal area", "unspecified topic"],
        "topic": "unspecified topic",
        "subtopic": "unspecified subtopic",
        "micro_topic": "requires manual review",
        "concepts_en": ["manual review required", "legal citation", "source text available"],
        "terms_original": terms,
        "statute_anchors": statutes,
        "case_anchors": cases,
        "doctrinal_rule": "",
        "legal_test": "",
        "fact_pattern_tags": [],
        "procedural_context": "",
        "paragraph_role": "neutral",
        "authority_role": ["none"],
        "outcome_signal": "none",
        "specificity_score": 0.0,
        "_fallback_reason": str(error)[:500],
    }
    return fallback

def build_retrieval_views(citation: str, enr: Dict[str, Any], text: str) -> Dict[str, str]:
    court_base = citation_base(citation)
    semantic_parts = (
        [enr.get("legal_area", ""), enr.get("topic", ""), enr.get("subtopic", ""), enr.get("micro_topic", "")]
        + list(enr.get("concepts_en", []))
        + list(enr.get("fact_pattern_tags", []))
    )
    semantic = " ".join(dedupe_keep_order(semantic_parts))

    return {
        "semantic_concepts_en": semantic,
        "topic_path": " > ".join(dedupe_keep_order(enr.get("legal_domain_path", []))),
        "original_terms_view": " ".join(dedupe_keep_order(enr.get("terms_original", []))),
        "statute_anchor_view": " ".join(dedupe_keep_order(enr.get("statute_anchors", []) + enr.get("concepts_en", []))),
        "case_anchor_view": " ".join(dedupe_keep_order([citation, court_base] + enr.get("case_anchors", []))),
        "legal_rule_view": " ".join(dedupe_keep_order([enr.get("doctrinal_rule", ""), enr.get("legal_test", "")])),
        "fact_pattern_view": " ".join(dedupe_keep_order(enr.get("fact_pattern_tags", []))),
        "raw_context": trim_text(text, 1200),
    }


In [ ]:
# Cell 6 - vLLM engine with safe JSON mode

class VllmJsonEngine:
    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.tokenizer = None
        self.llm = None
        self.SamplingParams = None
        self.structured_kwargs_mode = "disabled"

        if cfg.force_triton_attention:
            # Some vLLM versions warn about this env var when attention_config is also passed.
            # It is harmless, but attention_config below is the real setting.
            os.environ["VLLM_ATTENTION_BACKEND"] = "TRITON_ATTN"

        from transformers import AutoTokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(cfg.model_name, trust_remote_code=True)

        from vllm import LLM, SamplingParams
        self.SamplingParams = SamplingParams

        llm_kwargs = dict(
            model=cfg.model_name,
            trust_remote_code=True,
            tensor_parallel_size=cfg.tensor_parallel_size,
            gpu_memory_utilization=cfg.gpu_memory_utilization,
            max_model_len=cfg.max_model_len,
            enforce_eager=cfg.enforce_eager,
            max_num_seqs=cfg.max_num_seqs,
            quantization=cfg.quantization,
            disable_custom_all_reduce=cfg.disable_custom_all_reduce,
            disable_log_stats=True,
        )

        attn_config = self._build_attention_config() if cfg.force_triton_attention else None
        if attn_config is not None:
            llm_kwargs["attention_config"] = attn_config
            print("[vLLM] using explicit AttentionConfig backend=TRITON_ATTN")

        self.llm = LLM(**llm_kwargs)

        if cfg.use_structured_outputs:
            self.structured_kwargs_mode = self._detect_structured_mode()
        else:
            self.structured_kwargs_mode = "disabled"

        print("Engine: vLLM")
        print("Structured decoding mode:", self.structured_kwargs_mode)
        print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES", "<all>"))
        print("tensor_parallel_size:", cfg.tensor_parallel_size)

    def _build_attention_config(self):
        for import_path in ["vllm.config", "vllm.config.attention"]:
            try:
                mod = __import__(import_path, fromlist=["AttentionConfig"])
                AttentionConfig = getattr(mod, "AttentionConfig")
                try:
                    return AttentionConfig(backend="TRITON_ATTN")
                except Exception:
                    return AttentionConfig(backend="triton_attn")
            except Exception:
                continue
        return None

    def _detect_structured_mode(self) -> str:
        import inspect
        sig = inspect.signature(self.SamplingParams)
        params = sig.parameters
        if "guided_json" in params:
            return "guided_json"
        if "guided_decoding" in params:
            return "guided_decoding"
        if "structured_outputs" in params:
            # This mode caused AttributeError("'dict' object has no attribute '_backend'")
            # in your Kaggle v0.20 run, so it is only used if explicitly enabled and no
            # safer guided_json/guided_decoding path exists.
            return "structured_outputs"
        return "none"

    def _sampling_params(self, max_tokens: int, structured: bool = True):
        kwargs = dict(
            temperature=self.cfg.temperature,
            top_p=self.cfg.top_p,
            max_tokens=max_tokens,
            repetition_penalty=self.cfg.repetition_penalty,
        )

        if structured and self.cfg.use_structured_outputs:
            mode = self.structured_kwargs_mode
            if mode == "guided_json":
                kwargs["guided_json"] = ENRICHMENT_JSON_SCHEMA
            elif mode == "guided_decoding":
                try:
                    from vllm.sampling_params import GuidedDecodingParams
                    kwargs["guided_decoding"] = GuidedDecodingParams(json=ENRICHMENT_JSON_SCHEMA)
                except Exception:
                    # If construction fails, silently fall back to prompt-only JSON.
                    pass
            elif mode == "structured_outputs":
                # Disabled by default because this broke in the reported Kaggle environment.
                # Try only if cfg.use_structured_outputs=True.
                try:
                    from vllm import StructuredOutputsParams
                    # Different vLLM builds disagree on constructor shape.
                    # Prefer no structured_outputs if incompatible.
                    try:
                        kwargs["structured_outputs"] = StructuredOutputsParams(json_schema=ENRICHMENT_JSON_SCHEMA)
                    except TypeError:
                        kwargs["structured_outputs"] = StructuredOutputsParams(json=ENRICHMENT_JSON_SCHEMA)
                except Exception:
                    pass

        return self.SamplingParams(**kwargs)

    def _chat_to_prompt(self, user_prompt: str) -> str:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ]
        try:
            return self.tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=self.cfg.enable_thinking,
            )
        except TypeError:
            return self.tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )

    def generate_prompts(self, prompts: List[str], max_tokens: int, structured: bool = True) -> List[str]:
        full_prompts = [self._chat_to_prompt(p) for p in prompts]
        sampling_params = self._sampling_params(max_tokens=max_tokens, structured=structured)
        outputs = self.llm.generate(full_prompts, sampling_params, use_tqdm=False)
        texts = []
        for out in outputs:
            try:
                texts.append(out.outputs[0].text)
            except Exception:
                texts.append("")
        return texts

class TransformersJsonEngine:
    """Fallback engine. Slower than vLLM but useful if vLLM fails."""
    def __init__(self, cfg: Config):
        self.cfg = cfg
        from transformers import AutoTokenizer, AutoModelForCausalLM
        self.tokenizer = AutoTokenizer.from_pretrained(cfg.model_name, trust_remote_code=True)
        self.model = AutoModelForCausalLM.from_pretrained(
            cfg.model_name,
            trust_remote_code=True,
            device_map="auto",
            torch_dtype="auto",
        )
        self.model.eval()
        print("Engine: Transformers")

    def _chat_to_prompt(self, user_prompt: str) -> str:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ]
        try:
            return self.tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=self.cfg.enable_thinking,
            )
        except TypeError:
            return self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    def generate_prompts(self, prompts: List[str], max_tokens: int, structured: bool = True) -> List[str]:
        outs = []
        for p in prompts:
            full = self._chat_to_prompt(p)
            inputs = self.tokenizer(full, return_tensors="pt").to(self.model.device)
            with torch.no_grad():
                gen = self.model.generate(
                    **inputs,
                    max_new_tokens=max_tokens,
                    do_sample=False,
                    repetition_penalty=self.cfg.repetition_penalty,
                    pad_token_id=self.tokenizer.eos_token_id,
                )
            new_tokens = gen[0, inputs["input_ids"].shape[1]:]
            outs.append(self.tokenizer.decode(new_tokens, skip_special_tokens=True))
        return outs

def load_engine(cfg: Config):
    if cfg.engine.lower() == "vllm":
        return VllmJsonEngine(cfg)
    elif cfg.engine.lower() == "transformers":
        return TransformersJsonEngine(cfg)
    else:
        raise ValueError("cfg.engine must be 'vllm' or 'transformers'")

engine = load_engine(cfg)


In [ ]:
# Cell 7 - Enrichment runner with retry, repair, validation, and fallback

def make_row_dict(row: pd.Series) -> Dict[str, Any]:
    return {
        "_source_row": int(row["_source_row"]),
        "citation": str(row[citation_col]),
        "text": str(row[text_col]),
    }

def enrich_one_from_raw(row: Dict[str, Any], raw: str) -> Tuple[Optional[Dict[str, Any]], List[str]]:
    obj = parse_json_lenient(raw)
    enr = normalize_enrichment(obj, citation=row["citation"], text=row["text"])
    errors = validate_enrichment(enr)
    if errors:
        return None, errors
    return enr, []

def try_enrich_single(row: Dict[str, Any], first_raw: Optional[str] = None) -> Dict[str, Any]:
    attempts = []
    raw = first_raw

    for attempt in range(cfg.max_retries + 1):
        try:
            if raw is None:
                prompt = build_prompt(row, cfg, repair=False)
                raw = engine.generate_prompts([prompt], max_tokens=cfg.max_new_tokens, structured=True)[0]

            enr, errors = enrich_one_from_raw(row, raw)
            if enr is not None:
                return {
                    "enrichment": enr,
                    "status": "ok" if attempt == 0 else "ok_after_retry",
                    "attempts": attempts,
                    "raw_output": raw,
                }

            raise ValueError(";".join(errors))

        except Exception as exc:
            err = repr(exc)
            attempts.append({"attempt": attempt, "error": err, "raw_output": (raw or "")[:2500]})

            # Prepare repair retry. Use unstructured if structured mode somehow keeps failing/truncating.
            if attempt < cfg.max_retries:
                repair_prompt = build_prompt(
                    row,
                    cfg,
                    repair=True,
                    bad_output=raw or "",
                    error=err,
                )
                try:
                    raw = engine.generate_prompts(
                        [repair_prompt],
                        max_tokens=cfg.retry_max_new_tokens,
                        structured=True,
                    )[0]
                except Exception:
                    raw = engine.generate_prompts(
                        [repair_prompt],
                        max_tokens=cfg.retry_max_new_tokens,
                        structured=False,
                    )[0]
            else:
                fallback = deterministic_fallback(row["citation"], row["text"], err)
                return {
                    "enrichment": fallback,
                    "status": "fallback_after_failure",
                    "attempts": attempts,
                    "raw_output": raw or "",
                }

def build_output_record(row: Dict[str, Any], result: Dict[str, Any]) -> Dict[str, Any]:
    enr = result["enrichment"]
    citation = row["citation"]
    text = row["text"]
    rec = {
        "_source_row": row["_source_row"],
        "citation": citation,
        "court_base": citation_base(citation),
        "source_family": "court",
        "text": text,
        "rag_enrichment": enr,
        "retrieval_views": build_retrieval_views(citation, enr, text),
        "enrichment_quality": {
            "method": "llm_compact_legal_descriptor_enrichment" if result["status"] != "fallback_after_failure" else "deterministic_fallback",
            "generation_status": result["status"],
            "question_generation_used": False,
            "summary_generation_used": False,
            "grounded_references_only": True,
            "has_specific_topic": bool(enr.get("topic") and enr.get("subtopic") and enr.get("micro_topic")),
            "has_original_language_terms": bool(enr.get("terms_original")),
            "has_statute_anchor": bool(enr.get("statute_anchors")),
            "has_case_anchor": bool(enr.get("case_anchors")),
            "low_value_paragraph": len(str(text).strip()) < 120,
            "attempt_count": len(result.get("attempts", [])) + 1,
        },
    }
    if result["status"] == "fallback_after_failure":
        rec["_debug_attempts"] = result.get("attempts", [])
    return rec

out_dir = Path(cfg.output_dir)
out_dir.mkdir(parents=True, exist_ok=True)
out_jsonl = out_dir / cfg.output_jsonl
out_csv = out_dir / cfg.output_preview_csv
out_failures = out_dir / cfg.output_failures_jsonl

records = []
failure_records = []
t0 = time.time()

# First pass in mini-batches.
for start in tqdm(range(0, len(work_df), cfg.batch_size), desc="first-pass batches"):
    batch_df = work_df.iloc[start:start + cfg.batch_size]
    rows = [make_row_dict(row) for _, row in batch_df.iterrows()]
    prompts = [build_prompt(row, cfg, repair=False) for row in rows]

    try:
        raws = engine.generate_prompts(prompts, max_tokens=cfg.max_new_tokens, structured=True)
    except Exception as batch_exc:
        print("Batch generation failed; falling back to single-row generation:", repr(batch_exc))
        raws = [None] * len(rows)

    for row, raw in zip(rows, raws):
        result = try_enrich_single(row, first_raw=raw)
        rec = build_output_record(row, result)
        records.append(rec)
        if result["status"] == "fallback_after_failure":
            failure_records.append(rec)

elapsed = time.time() - t0

# Write JSONL.
with out_jsonl.open("w", encoding="utf-8") as f:
    for rec in records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

# Flatten preview CSV.
preview_rows = []
for rec in records:
    enr = rec["rag_enrichment"]
    views = rec["retrieval_views"]
    q = rec["enrichment_quality"]
    preview_rows.append({
        "_source_row": rec["_source_row"],
        "citation": rec["citation"],
        "court_base": rec["court_base"],
        "generation_status": q["generation_status"],
        "legal_area": enr.get("legal_area"),
        "topic": enr.get("topic"),
        "subtopic": enr.get("subtopic"),
        "micro_topic": enr.get("micro_topic"),
        "concepts_en": " | ".join(enr.get("concepts_en", [])),
        "terms_original": " | ".join(enr.get("terms_original", [])),
        "statute_anchors": " | ".join(enr.get("statute_anchors", [])),
        "case_anchors": " | ".join(enr.get("case_anchors", [])),
        "doctrinal_rule": enr.get("doctrinal_rule"),
        "legal_test": enr.get("legal_test"),
        "fact_pattern_tags": " | ".join(enr.get("fact_pattern_tags", [])),
        "procedural_context": enr.get("procedural_context"),
        "paragraph_role": enr.get("paragraph_role"),
        "authority_role": " | ".join(enr.get("authority_role", [])),
        "outcome_signal": enr.get("outcome_signal"),
        "specificity_score": enr.get("specificity_score"),
        "semantic_concepts_en": views.get("semantic_concepts_en"),
        "original_terms_view": views.get("original_terms_view"),
    })

preview_df = pd.DataFrame(preview_rows)
preview_df.to_csv(out_csv, index=False)

if failure_records:
    with out_failures.open("w", encoding="utf-8") as f:
        for rec in failure_records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
else:
    # Remove stale failure file from previous runs if present.
    if out_failures.exists():
        out_failures.unlink()

status_counts = Counter(r["enrichment_quality"]["generation_status"] for r in records)

print(f"Generated {len(records)} rows in {elapsed:.1f}s")
print(f"Rows/s: {len(records)/max(elapsed, 1e-9):.3f}")
print("Status counts:", dict(status_counts))
print("JSONL:", out_jsonl)
print("Preview CSV:", out_csv)
if failure_records:
    print("Fallback/failure records:", out_failures)

display(preview_df)


In [ ]:
# Cell 8 - Inspect full JSON for one record

if records:
    print(json.dumps(records[0], ensure_ascii=False, indent=2)[:8000])
else:
    print("No records generated.")


In [ ]:
# Cell 9 - Quality checks

def check_no_forbidden_fields(rec: Dict[str, Any]) -> List[str]:
    forbidden = {"query_phrases_en", "summary_en", "natural_language_queries", "english_summary"}
    hits = []
    def walk(x, path=""):
        if isinstance(x, dict):
            for k, v in x.items():
                if k in forbidden:
                    hits.append(path + "." + k if path else k)
                walk(v, path + "." + k if path else k)
        elif isinstance(x, list):
            for i, v in enumerate(x):
                walk(v, f"{path}[{i}]")
    walk(rec)
    return hits

qc_rows = []
for rec in records:
    enr = rec["rag_enrichment"]
    qc_rows.append({
        "citation": rec["citation"],
        "status": rec["enrichment_quality"]["generation_status"],
        "forbidden_fields": check_no_forbidden_fields(rec),
        "concept_count": len(enr.get("concepts_en", [])),
        "terms_original_count": len(enr.get("terms_original", [])),
        "self_case_anchor_present": rec["citation"] in enr.get("case_anchors", []) or rec["court_base"] in enr.get("case_anchors", []),
        "specificity_score": enr.get("specificity_score"),
    })

qc_df = pd.DataFrame(qc_rows)
display(qc_df)

print("Forbidden field rows:", int(qc_df["forbidden_fields"].apply(bool).sum()))
print("Self case anchor rows:", int(qc_df["self_case_anchor_present"].sum()))
print("Fallback rows:", int((qc_df["status"] == "fallback_after_failure").sum()))
